<a href="https://colab.research.google.com/github/yogi910910-hash/html-quick-styler/blob/main/HTML_Quick_Styler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 HTML Quick-Styler
### AI-Powered HTML Page Generator with IBM Granite 3.3 2B
> Make sure Runtime → Change Runtime Type → **GPU (T4)** is selected before running!

In [1]:
# CELL 1: Install dependencies
!pip install -q gradio transformers torch accelerate huggingface_hub
print('✅ Dependencies installed')

✅ Dependencies installed


In [12]:
# Download the py file from GitHub
!wget https://raw.githubusercontent.com/yogi910910-hash/html-quick-styler/main/html_quick_styler.py

# Import everything from it
from html_quick_styler import *

--2026-03-07 07:20:53--  https://raw.githubusercontent.com/yogi910910-hash/html-quick-styler/main/html_quick_styler.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35223 (34K) [text/plain]
Saving to: ‘html_quick_styler.py.1’

html_quick_styler.p 100%[===================>]  34.40K  --.-KB/s    in 0.002s  

2026-03-07 07:20:53 (21.5 MB/s) - ‘html_quick_styler.py.1’ saved [35223/35223]



In [2]:
# CELL 2: Imports
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re, json
print('✅ Imports done')

✅ Imports done


In [3]:
# CELL 3: Load IBM Granite 3.3 2B
MODEL_ID = 'ibm-granite/granite-3.3-2b-instruct'
print('Loading model... (this takes ~2 min on first run)')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)
print(f'✅ Model loaded on: {next(model.parameters()).device}')

Loading model... (this takes ~2 min on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Model loaded on: cuda:0


In [4]:
# CELL 4: Color Palette Generator
class ColorPaletteGenerator:
    PALETTES = {
        'modern':   {'primary':'#6366F1','secondary':'#8B5CF6','accent':'#06B6D4','background':'#0F172A','surface':'#1E293B','text_primary':'#F8FAFC','text_secondary':'#94A3B8','border':'#334155'},
        'vibrant':  {'primary':'#FF6B6B','secondary':'#4ECDC4','accent':'#FFE66D','background':'#1A1A2E','surface':'#16213E','text_primary':'#FFFFFF','text_secondary':'#B0B0C3','border':'#2D2D44'},
        'ocean':    {'primary':'#0077B6','secondary':'#00B4D8','accent':'#90E0EF','background':'#03045E','surface':'#023E8A','text_primary':'#CAF0F8','text_secondary':'#90E0EF','border':'#0077B6'},
        'forest':   {'primary':'#2D6A4F','secondary':'#52B788','accent':'#B7E4C7','background':'#081C15','surface':'#1B4332','text_primary':'#D8F3DC','text_secondary':'#95D5B2','border':'#2D6A4F'},
        'sunset':   {'primary':'#F77F00','secondary':'#FCBF49','accent':'#EAE2B7','background':'#1A0000','surface':'#2D0A00','text_primary':'#FFF3E0','text_secondary':'#FFCC80','border':'#BF360C'},
        'luxury':   {'primary':'#C9A84C','secondary':'#B8860B','accent':'#FFD700','background':'#0D0D0D','surface':'#1A1A1A','text_primary':'#F5F5DC','text_secondary':'#D4AF37','border':'#3D3D00'},
        'creative': {'primary':'#E040FB','secondary':'#7C4DFF','accent':'#00E5FF','background':'#12005E','surface':'#1A0070','text_primary':'#EDE7F6','text_secondary':'#CE93D8','border':'#4A148C'},
        'minimal':  {'primary':'#212121','secondary':'#424242','accent':'#757575','background':'#FAFAFA','surface':'#FFFFFF','text_primary':'#212121','text_secondary':'#757575','border':'#E0E0E0'},
    }
    @classmethod
    def get_palette(cls, style): return cls.PALETTES.get(style.lower(), cls.PALETTES['modern'])
    @classmethod
    def get_all_palettes(cls): return cls.PALETTES
    @classmethod
    def generate_palette_preview_html(cls):
        html = "<div style='display:grid;grid-template-columns:repeat(auto-fill,minmax(220px,1fr));gap:16px;padding:16px'>"
        for name, colors in cls.PALETTES.items():
            swatches = ''.join(f"<div style='width:28px;height:28px;border-radius:50%;background:{v};border:2px solid #ffffff33'></div>" for v in colors.values())
            html += f"<div style='background:{colors['surface']};border:1px solid {colors['border']};border-radius:12px;padding:16px'><div style='font-size:16px;font-weight:700;color:{colors['text_primary']};margin-bottom:8px;text-transform:capitalize'>{name}</div><div style='display:flex;gap:6px;flex-wrap:wrap'>{swatches}</div><div style='font-size:11px;color:{colors['text_secondary']};margin-top:8px'>Primary: {colors['primary']}</div></div>"
        return html + '</div>'
print('✅ ColorPaletteGenerator ready')

✅ ColorPaletteGenerator ready


In [5]:
# CELL 5: HTML Page Builder (paste full class from html_quick_styler.py)
# Copy the HTMLPageBuilder class here
print('Paste the HTMLPageBuilder class from html_quick_styler.py into this cell')

Paste the HTMLPageBuilder class from html_quick_styler.py into this cell


In [6]:
# CELL 6: CSS Generator (paste generate_css function from html_quick_styler.py)
print('Paste generate_css() from html_quick_styler.py into this cell')

Paste generate_css() from html_quick_styler.py into this cell


In [7]:
# CELL 7: AI Enhancement
def ai_enhance_content(title, description, section):
    prompt = f'Generate a compelling {section} headline for: {title}. Respond ONLY with JSON: {{"headline": "..."}}'
    try:
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        match = re.search(r'\{.*?\}', generated, re.DOTALL)
        if match: return json.loads(match.group()).get('headline', title)
    except: pass
    return title
print('✅ AI enhancement ready')

✅ AI enhancement ready


In [8]:
# CELL 8: Page Assembly (paste generate_page() from html_quick_styler.py)
print('Paste generate_page() from html_quick_styler.py into this cell')

Paste generate_page() from html_quick_styler.py into this cell


In [11]:
# CELL 9 + 10: Gradio Interface & Launch
# Paste create_interface() and the final launch lines from html_quick_styler.py
demo = create_interface()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://823a885446cce7a531.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://823a885446cce7a531.gradio.live
